In [ ]:
let
    // Connect to SharePoint site (parameterized)
    Source = SharePoint.Tables(pSharePointSite, [Implementation = "2.0", ViewMode = "Default"]),

    // Select the list dynamically using parameter
    ListData = Source{[Title = pListName]}[Items],

    // Expand common SharePoint person fields
    #"Expanded Created By" = Table.ExpandListColumn(ListData, "Created By"),
    #"Expanded Created By 1" = Table.ExpandRecordColumn(
        #"Expanded Created By",
        "Created By",
        {"title"},
        {"Created By.title"}
    ),

    #"Expanded Modified By" = Table.ExpandListColumn(#"Expanded Created By 1", "Modified By"),
    #"Expanded Modified By 1" = Table.ExpandRecordColumn(
        #"Expanded Modified By",
        "Modified By",
        {"title"},
        {"Modified By.title"}
    ),

    // Rename common source metadata columns
    #"Renamed Common Source Metadata Columns" = Table.RenameColumns(
        #"Expanded Modified By 1",
        {
            {"Created", "z_src_created_date_time"},
            {"Created By.title", "z_src_created_by_user"},
            {"Modified", "z_src_modified_date_time"},
            {"Modified By.title", "z_src_modified_by_user"}
        }
    ),

    // Set data types for common source metadata columns
    #"Changed Common Metadata Types" = Table.TransformColumnTypes(
        #"Renamed Common Source Metadata Columns",
        {
            {"z_src_created_date_time", type datetime},
            {"z_src_created_by_user", type text},
            {"z_src_modified_date_time", type datetime},
            {"z_src_modified_by_user", type text}
        }
    )
in
    #"Changed Common Metadata Types"